# EEGNet Interpretability Notebook for Thesis Figures

This notebook generates EEGNet-specific interpretability artifacts for the thesis.

It is designed for the **grouped within-subject benchmark** and works by:

1. building the grouped benchmark dataset from the project pipeline
2. training or reloading **one representative EEGNet model**
3. interpreting that model on its **held-out evaluation fold**
4. saving all figures and tables to:

- `result/thesis_artifacts/figures`
- `result/thesis_artifacts/tables`

## Main outputs

### Figures
1. **Broadband EEGNet interpretability summary**
   - time-importance curve
   - top channels
   - P300-window scalp topography

2. **Band-limited EEGNet diagnostic**
   - theta / alpha / gamma time-importance curves
   - theta / alpha / gamma scalp topographies

3. **Trial-consistency summary**
   - trial-by-time attribution heatmap
   - mean attribution across selected held-out trials

### Tables
- broadband top channels
- broadband time importance
- broadband peak times
- band importance summary
- band top channels
- trial attribution summary
- run metadata

## Important note

This notebook is intentionally focused on **one evaluation setting** and **one held-out fold** so that the model, predictions, and interpretability results are all fully aligned.

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import mne

from scipy.signal import butter, sosfiltfilt

from oddball_eeg.bids import make_epoching_config_from_experiment
from oddball_eeg.config import get_experiment_family
from oddball_eeg.io import save_dataframe, save_experiment_metadata, save_json
from oddball_eeg.loader import load_subjects_grouped
from oddball_eeg.metrics import classification_metric_dict
from oddball_eeg.models import get_model
from oddball_eeg.preprocessing import (
    apply_crop_to_loaded_dataset,
    compute_channel_normalization_stats,
    normalize_epochs,
)
from oddball_eeg.reproducibility import get_default_device
from oddball_eeg.splits import build_folds_by_subject
from oddball_eeg.training import train_eegnet_one_fold
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
    valid_topomap_picks,
)


warnings.filterwarnings("ignore")

set_pandas_display_options()

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 11

## Configuration

In [ ]:
# Change only this line when switching datasets.
# EXPERIMENT_NAME = "ds005863_within_subject_main"
EXPERIMENT_NAME = "ds003061_within_subject_main"

experiment_family = get_experiment_family(EXPERIMENT_NAME)
cfg = experiment_family["main"]

DEVICE = get_default_device()

# One representative grouped within-subject fold to interpret.
REPRESENTATIVE_SUBJECT = "006"
REPRESENTATIVE_FOLD = 1
REPRESENTATIVE_SEED = 42
FORCE_RETRAIN = False

# Interpretability settings
POSITIVE_LABEL = 1
USE_ONLY_CORRECT = True
ATTR_BATCH_SIZE = 16
IG_STEPS = 32
BASELINE_MODE = "prestim_mean"  # "zeros" or "prestim_mean"

PRESTIM_WINDOW = (-0.2, 0.0)
P300_WINDOW = (0.25, 0.50)

BANDS = {
    "theta": (4.0, 8.0),
    "alpha": (8.0, 12.0),
    "gamma": (30.0, 45.0),
}

OUT_DIR = cfg.artifact_dir("eegnet_interpretability")
FIG_DIR = cfg.artifact_dir("eegnet_interpretability", "figures")
TABLE_DIR = cfg.artifact_dir("eegnet_interpretability", "tables")
CHECKPOINT_DIR = cfg.checkpoint_dir("eegnet_interpretability")
DEBUG_DIR = cfg.artifact_dir("eegnet_interpretability", "debug")

RUN_INFO_JSON = TABLE_DIR / "eegnet_interpretability_run_info.json"

save_experiment_metadata(
    cfg,
    extra={
        "notebook": "11_eegnet_interpretability.ipynb",
        "purpose": "EEGNet integrated-gradient interpretability for one grouped within-subject held-out fold.",
        "experiment_key": EXPERIMENT_NAME,
        "dataset": cfg.dataset.dataset,
        "task": cfg.dataset.task,
        "signal": cfg.signal.slug(),
        "representative_subject": REPRESENTATIVE_SUBJECT,
        "representative_fold": REPRESENTATIVE_FOLD,
        "representative_seed": REPRESENTATIVE_SEED,
        "positive_label": POSITIVE_LABEL,
        "use_only_correct": USE_ONLY_CORRECT,
        "ig_steps": IG_STEPS,
        "baseline_mode": BASELINE_MODE,
        "prestim_window": list(PRESTIM_WINDOW),
        "p300_window": list(P300_WINDOW),
        "bands": BANDS,
        "artifact_dir": str(OUT_DIR),
        "figure_dir": str(FIG_DIR),
        "table_dir": str(TABLE_DIR),
        "checkpoint_dir": str(CHECKPOINT_DIR),
    },
)

print("Experiment key :", EXPERIMENT_NAME)
print("Dataset        :", cfg.dataset.dataset)
print("Task           :", cfg.dataset.task)
print("Signal config  :", cfg.signal)
print("Device         :", DEVICE)
print("Artifacts      :", OUT_DIR)
print("Figures        :", FIG_DIR)
print("Tables         :", TABLE_DIR)
print("Checkpoints    :", CHECKPOINT_DIR)

## Build grouped benchmark dataset from the project pipeline

In [ ]:
def build_grouped_dataset_and_folds():
    epoching_cfg = make_epoching_config_from_experiment(cfg)

    print("Epoching config:")
    print(epoching_cfg)

    data_full = load_subjects_grouped(
        dataset=cfg.dataset.dataset,
        task_name=cfg.dataset.task,
        subjects=cfg.subjects.subjects,
        epoching_cfg=epoching_cfg,
        chunk_size=cfg.splits.group_chunk_size,
        apply_baseline=True,
        verbose=True,
    )

    crop_window = cfg.effective_crop_window()
    assert crop_window is not None, "Expected an active crop window."

    data_main = apply_crop_to_loaded_dataset(
        data_full,
        tmin=crop_window[0],
        tmax=crop_window[1],
    )

    folds_by_sub = build_folds_by_subject(
        data=data_main,
        n_splits=cfg.splits.n_splits,
        purge_neighbors=cfg.splits.purge_neighbor_groups,
    )

    return data_main, folds_by_sub


data_main, folds_by_sub = build_grouped_dataset_and_folds()

times = data_main.times.astype(np.float32)
ch_names = list(data_main.ch_names)
sfreq = float(data_main.sfreq)

print("Subjects  :", data_main.subjects)
print("n_channels:", len(ch_names))
print("n_times   :", len(times))
print("sfreq     :", sfreq)
print("device    :", DEVICE)

## Train or reload one representative EEGNet model and recover its held-out test fold

This cell trains or reloads one representative EEGNet model for a chosen subject and fold.

Interpretability will then be computed on the **actual held-out test fold** used for that model.

In [ ]:
def checkpoint_path_for(subject: str, fold: int, seed: int) -> Path:
    return CHECKPOINT_DIR / f"eegnet_grouped_sub-{subject}_fold-{fold}_seed-{seed}.pt"


def build_eegnet_kwargs() -> dict:
    return {
        "n_classes": cfg.eegnet.n_classes,
        "f1": cfg.eegnet.f1,
        "d": cfg.eegnet.d,
        "f2": cfg.eegnet.f2,
        "k1": cfg.eegnet.k1,
        "k2": cfg.eegnet.k2,
        "dropout": cfg.eegnet.dropout,
    }


def load_checkpoint_model(ckpt_path: Path):
    ckpt = torch.load(ckpt_path, map_location="cpu")

    model = get_model(
        "eegnet",
        n_ch=int(ckpt["n_ch"]),
        n_times=int(ckpt["n_times"]),
        eegnet_kwargs=ckpt["eegnet_kwargs"],
    )

    model.load_state_dict(ckpt["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    return ckpt, model


def train_or_load_representative_eegnet(
        subject: str,
        fold: int,
        seed: int,
        force_retrain: bool = False,
):
    assert subject in data_main.subjects, f"Unknown subject: {subject}"

    X_sub_raw = data_main.X_by_sub[subject].astype(np.float32)
    y_sub = data_main.y_by_sub[subject].astype(np.int64)
    g_sub = data_main.groups_by_sub[subject].astype(np.int64)

    n_folds = len(folds_by_sub[subject])
    assert 1 <= fold <= n_folds, f"Fold must be in 1..{n_folds}, got {fold}"

    tr_idx, te_idx = folds_by_sub[subject][fold - 1]

    X_train_raw = X_sub_raw[tr_idx].astype(np.float32)
    y_train = y_sub[tr_idx].astype(np.int64)
    g_train = g_sub[tr_idx].astype(np.int64)

    X_eval_raw = X_sub_raw[te_idx].astype(np.float32)
    y_eval = y_sub[te_idx].astype(np.int64)
    groups_eval = g_sub[te_idx].astype(np.int64)
    subjects_eval = np.array([subject] * len(y_eval), dtype=object)

    # This is the critical alignment step:
    # EEGNet was trained with train-fold channel normalization.
    norm_stats = compute_channel_normalization_stats(X_train_raw)

    eegnet_kwargs = build_eegnet_kwargs()
    ckpt_path = checkpoint_path_for(subject, fold, seed)

    if ckpt_path.exists() and not force_retrain:
        ckpt, model = load_checkpoint_model(ckpt_path)

        run_info = {
            "subject": ckpt["subject"],
            "fold": ckpt["fold"],
            "seed": ckpt["seed"],
            "best_epoch": ckpt.get("best_epoch"),
            "best_val_auc": ckpt.get("best_val_auc"),
            "loaded_from_checkpoint": True,
        }

    else:
        result = train_eegnet_one_fold(
            X_train_fold=X_train_raw,
            y_train_fold=y_train,
            groups_train_fold=g_train,
            X_test_fold=X_eval_raw,
            y_test_fold=y_eval,
            device=DEVICE,
            normalize=cfg.eegnet.normalize,
            batch_size=cfg.eegnet.batch_size,
            lr=cfg.eegnet.lr,
            weight_decay=cfg.eegnet.weight_decay,
            epochs=cfg.eegnet.epochs,
            patience=cfg.eegnet.early_stopping_patience,
            val_size=cfg.splits.val_size,
            val_n_tries=cfg.splits.val_n_tries,
            use_class_weight=cfg.eegnet.use_class_weight,
            grad_clip_norm=cfg.eegnet.grad_clip_norm,
            eegnet_kwargs=eegnet_kwargs,
            seed=seed,
            verbose=True,
        )

        model = result["model"]
        model.to(DEVICE)
        model.eval()

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_name": "eegnet",
                "subject": subject,
                "fold": int(fold),
                "seed": int(seed),
                "n_ch": int(X_sub_raw.shape[1]),
                "n_times": int(X_sub_raw.shape[2]),
                "eegnet_kwargs": eegnet_kwargs,
                "best_epoch": result["best_epoch"],
                "best_val_auc": result["best_val_auc"],
                "normalization_mean": norm_stats.mean,
                "normalization_std": norm_stats.std,
            },
            ckpt_path,
        )

        run_info = {
            "subject": subject,
            "fold": fold,
            "seed": seed,
            "best_epoch": result["best_epoch"],
            "best_val_auc": result["best_val_auc"],
            "loaded_from_checkpoint": False,
        }

    X_eval_norm = normalize_epochs(X_eval_raw, norm_stats)

    return {
        "model": model,
        "ckpt_path": ckpt_path,
        "X_train_raw": X_train_raw,
        "X_eval_raw": X_eval_raw,
        "X_eval_norm": X_eval_norm,
        "y_eval": y_eval,
        "subjects_eval": subjects_eval,
        "groups_eval": groups_eval,
        "norm_stats": norm_stats,
        "run_info": run_info,
    }


rep = train_or_load_representative_eegnet(
    subject=REPRESENTATIVE_SUBJECT,
    fold=REPRESENTATIVE_FOLD,
    seed=REPRESENTATIVE_SEED,
    force_retrain=FORCE_RETRAIN,
)

model = rep["model"]
ckpt_path = rep["ckpt_path"]

X_eval_raw = rep["X_eval_raw"]
X_eval_norm = rep["X_eval_norm"]
y_eval = rep["y_eval"]
subjects_eval = rep["subjects_eval"]
groups_eval = rep["groups_eval"]
norm_stats = rep["norm_stats"]
run_info = rep["run_info"]

print("Checkpoint     :", ckpt_path)
print("Model          :", model.__class__.__name__)
print("Held-out subject:", run_info["subject"])
print("Held-out fold  :", run_info["fold"])
print("Best epoch     :", run_info["best_epoch"])
print("Best val AUC   :", run_info["best_val_auc"])
print("Loaded ckpt    :", run_info["loaded_from_checkpoint"])
print("Eval raw shape :", X_eval_raw.shape)
print("Eval norm shape:", X_eval_norm.shape)

## Prediction helpers

In [ ]:
def to_model_input(x_np: np.ndarray) -> torch.Tensor:
    """
    Expected input shape: (batch, channels, times)
    Converts to EEGNet shape: (batch, 1, channels, times)
    """
    x = torch.as_tensor(x_np, dtype=torch.float32)

    if x.ndim != 3:
        raise ValueError(f"Expected x with shape (batch, channels, times), got {x.shape}")

    return x.unsqueeze(1)


def extract_positive_logit(output: torch.Tensor) -> torch.Tensor:
    """
    Return the positive-class logit for attribution.
    """
    if isinstance(output, (tuple, list)):
        output = output[0]

    if output.ndim == 1:
        return output

    if output.ndim == 2 and output.shape[1] == 1:
        return output[:, 0]

    if output.ndim == 2 and output.shape[1] >= 2:
        return output[:, POSITIVE_LABEL]

    raise ValueError(f"Unsupported model output shape: {tuple(output.shape)}")


@torch.no_grad()
def predict_scores_and_labels(model, X_np: np.ndarray, batch_size: int = 64):
    logits_all = []

    model.eval()

    for start in range(0, len(X_np), batch_size):
        xb = X_np[start:start + batch_size]
        xb_t = to_model_input(xb).to(DEVICE)

        logits = model(xb_t)

        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        logits_all.append(logits.detach().cpu())

    logits_all = torch.cat(logits_all, dim=0)

    if logits_all.ndim == 2 and logits_all.shape[1] >= 2:
        probs_all = torch.softmax(logits_all, dim=1).numpy()
        probs = probs_all[:, POSITIVE_LABEL]
        preds = np.argmax(probs_all, axis=1).astype(int)
        scores = logits_all[:, POSITIVE_LABEL].numpy()
    else:
        scores = logits_all.reshape(-1).numpy()
        probs = 1.0 / (1.0 + np.exp(-scores))
        preds = (probs >= 0.5).astype(int)

    return scores, probs, preds

## Select held-out trials to interpret

By default:
- keep only **target-class trials**
- optionally keep only **correctly classified target trials**

In [ ]:
scores, probs, preds = predict_scores_and_labels(
    model,
    X_eval_norm,
    batch_size=128,
)

metrics_eval = classification_metric_dict(
    y_eval,
    probs,
    preds,
)

print("Unique y_eval labels       :", np.unique(y_eval))
print("POSITIVE_LABEL             :", POSITIVE_LABEL)
print("Total eval trials          :", len(y_eval))
print("Target trials              :", int((y_eval == POSITIVE_LABEL).sum()))
print("Correct predictions overall:", int((preds == y_eval).sum()))
print("Correct target predictions :", int(((y_eval == POSITIVE_LABEL) & (preds == y_eval)).sum()))
print("Eval AUC                   :", metrics_eval["auc"])
print("Eval balanced accuracy     :", metrics_eval["bal_acc"])

selection_reason = None

selection_mask = (y_eval == POSITIVE_LABEL)

if USE_ONLY_CORRECT:
    selection_mask = selection_mask & (preds == y_eval)

if selection_mask.sum() > 0:
    selection_reason = "target trials (correct only)" if USE_ONLY_CORRECT else "target trials"
else:
    print("\n[WARN] No trials found with the initial selection rule.")

    fallback_mask = (y_eval == POSITIVE_LABEL)
    if fallback_mask.sum() > 0:
        selection_mask = fallback_mask
        selection_reason = "target trials (fallback: include incorrect predictions)"
        print("[INFO] Falling back to all target trials.")
    else:
        fallback_mask = (preds == y_eval)
        if fallback_mask.sum() > 0:
            selection_mask = fallback_mask
            selection_reason = "all correct trials (fallback)"
            print("[INFO] Falling back to all correctly predicted trials.")
        else:
            selection_mask = np.ones(len(y_eval), dtype=bool)
            selection_reason = "all evaluation trials (last-resort fallback)"
            print("[INFO] Falling back to all evaluation trials.")

X_sel_raw = X_eval_raw[selection_mask]
X_sel_norm = X_eval_norm[selection_mask]

y_sel = y_eval[selection_mask]
subjects_sel = subjects_eval[selection_mask]
scores_sel = scores[selection_mask]
probs_sel = probs[selection_mask]
preds_sel = preds[selection_mask]

selection_summary = pd.DataFrame({
    "dataset": [cfg.dataset.dataset],
    "task": [cfg.dataset.task],
    "heldout_subject": [run_info["subject"]],
    "heldout_fold": [run_info["fold"]],
    "seed": [run_info["seed"]],
    "n_eval_trials": [len(X_eval_norm)],
    "n_selected_trials": [len(X_sel_norm)],
    "n_target_trials_total": [int((y_eval == POSITIVE_LABEL).sum())],
    "n_selected_subjects": [len(np.unique(subjects_sel)) if len(subjects_sel) > 0 else 0],
    "use_only_correct": [USE_ONLY_CORRECT],
    "positive_label": [POSITIVE_LABEL],
    "selection_reason": [selection_reason],
    "eval_auc": [metrics_eval["auc"]],
    "eval_pr_auc": [metrics_eval["pr_auc"]],
    "eval_bal_acc": [metrics_eval["bal_acc"]],
    "eval_acc": [metrics_eval["acc"]],
    "eval_f1": [metrics_eval["f1"]],
})

save_dataframe(
    selection_summary,
    TABLE_DIR / "eegnet_trial_selection_summary.csv",
)

print("\nSelection reason:", selection_reason)
print("Selected trials :", len(X_sel_norm))

display(round_numeric_columns(selection_summary, 4))

## Integrated gradients

In [ ]:
def get_window_mask(times, tmin, tmax):
    return (times >= tmin) & (times <= tmax)


def make_baseline_batch(x_batch_np: np.ndarray, times: np.ndarray) -> np.ndarray:
    """
    Baseline is built in the same normalized space as model input.
    """
    if BASELINE_MODE == "zeros":
        return np.zeros_like(x_batch_np, dtype=np.float32)

    if BASELINE_MODE == "prestim_mean":
        prestim_mask = get_window_mask(times, PRESTIM_WINDOW[0], PRESTIM_WINDOW[1])

        if prestim_mask.sum() == 0:
            raise ValueError("No samples found inside PRESTIM_WINDOW.")

        mean_prestim = x_batch_np[:, :, prestim_mask].mean(axis=2, keepdims=True)
        return np.repeat(mean_prestim, x_batch_np.shape[2], axis=2).astype(np.float32)

    raise ValueError(f"Unsupported BASELINE_MODE: {BASELINE_MODE}")


def integrated_gradients_batch(
        model,
        x_batch_np: np.ndarray,
        times: np.ndarray,
        steps: int = 32,
) -> np.ndarray:
    """
    Integrated gradients in normalized model-input space.

    Returns
    -------
    ig : np.ndarray
        Shape (batch, channels, times)
    """
    model.eval()

    x = to_model_input(x_batch_np).to(DEVICE)
    baseline = to_model_input(make_baseline_batch(x_batch_np, times)).to(DEVICE)

    alphas = torch.linspace(0.0, 1.0, steps, device=DEVICE)
    total_grad = torch.zeros_like(x)

    for alpha in alphas:
        x_interp = baseline + alpha * (x - baseline)
        x_interp.requires_grad_(True)

        out = model(x_interp)
        score = extract_positive_logit(out).sum()
        grad = torch.autograd.grad(score, x_interp)[0]

        total_grad += grad.detach()

    avg_grad = total_grad / steps
    ig = (x - baseline) * avg_grad

    return ig.squeeze(1).detach().cpu().numpy()

## Compute broadband EEGNet attributions

In [ ]:
if len(X_sel_norm) == 0:
    raise RuntimeError("No selected trials available for EEGNet interpretability.")

all_attr = []
all_subjects = []

for start in range(0, len(X_sel_norm), ATTR_BATCH_SIZE):
    xb = X_sel_norm[start:start + ATTR_BATCH_SIZE]
    sb = subjects_sel[start:start + ATTR_BATCH_SIZE]

    attr = integrated_gradients_batch(
        model,
        xb,
        times=times,
        steps=IG_STEPS,
    )

    all_attr.append(attr)
    all_subjects.append(np.asarray(sb))

attr_trials = np.concatenate(all_attr, axis=0)
attr_subjects = np.concatenate(all_subjects, axis=0)

abs_attr_trials = np.abs(attr_trials)
broadband_map = abs_attr_trials.mean(axis=0)

print("Attribution trials:", attr_trials.shape)
print("Broadband map     :", broadband_map.shape)

## Broadband summaries and tables

In [ ]:
p300_mask = get_window_mask(times, P300_WINDOW[0], P300_WINDOW[1])

time_importance = broadband_map.mean(axis=0)
channel_importance = broadband_map.mean(axis=1)
p300_channel_importance = broadband_map[:, p300_mask].mean(axis=1)

broadband_top_channels = (
    pd.DataFrame({
        "channel": ch_names,
        "importance_mean": p300_channel_importance,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

broadband_time_summary = pd.DataFrame({
    "time_s": times,
    "importance_mean": time_importance,
})

peak_idx = np.argsort(time_importance)[::-1][:15]

broadband_time_peaks = (
    pd.DataFrame({
        "time_s": times[peak_idx],
        "importance_mean": time_importance[peak_idx],
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

save_dataframe(
    broadband_top_channels,
    TABLE_DIR / "eegnet_broadband_top_channels.csv",
)

save_dataframe(
    broadband_time_summary,
    TABLE_DIR / "eegnet_broadband_time_importance.csv",
)

save_dataframe(
    broadband_time_peaks,
    TABLE_DIR / "eegnet_broadband_time_peaks.csv",
)

display(round_numeric_columns(broadband_top_channels.head(12), 6))

## Plot broadband EEGNet interpretability figure

In [ ]:
def make_eeg_info(ch_names, sfreq):
    info = mne.create_info(
        ch_names=ch_names,
        sfreq=sfreq,
        ch_types="eeg",
    )
    info.set_montage("standard_1020", on_missing="ignore")
    return info


info = make_eeg_info(ch_names, sfreq)
topomap_picks = valid_topomap_picks(info)
info_valid = mne.pick_info(info, topomap_picks, copy=True)


def plot_topomap_or_text(ax, values, info_valid, picks, title):
    try:
        values = np.asarray(values, dtype=float)
        values_valid = values[picks]

        mne.viz.plot_topomap(
            values_valid,
            info_valid,
            axes=ax,
            show=False,
            contours=6,
        )
        ax.set_title(title)

    except Exception as e:
        ax.text(
            0.5,
            0.5,
            f"Topomap unavailable\n{type(e).__name__}",
            ha="center",
            va="center",
        )
        ax.set_title(title)
        ax.axis("off")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: time importance
axes[0].plot(times, time_importance, linewidth=2)
axes[0].axvspan(P300_WINDOW[0], P300_WINDOW[1], alpha=0.2)
axes[0].axvline(0.0, linestyle="--", linewidth=1)
axes[0].set_title("EEGNet broadband time importance")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Mean |attribution|")
axes[0].grid(alpha=0.25)

# Panel 2: top channels
top_n = 12
top_df = broadband_top_channels.head(top_n).iloc[::-1]

axes[1].barh(top_df["channel"], top_df["importance_mean"])
axes[1].set_title(f"Top {top_n} channels in P300/ERP window")
axes[1].set_xlabel("Mean |attribution|")
axes[1].grid(axis="x", alpha=0.25)

# Panel 3: topomap
plot_topomap_or_text(
    axes[2],
    p300_channel_importance,
    info_valid,
    topomap_picks,
    f"Topography\n{P300_WINDOW[0]:.2f}–{P300_WINDOW[1]:.2f} s",
)

fig.suptitle(
    f"EEGNet broadband interpretability — {cfg.dataset.dataset}, {cfg.dataset.task}\n"
    f"Subject {REPRESENTATIVE_SUBJECT}, fold {REPRESENTATIVE_FOLD}",
    fontsize=14,
    y=1.03,
)

fig.tight_layout()

broadband_fig_path = FIG_DIR / "chapter7_eegnet_broadband_interpretability.png"

save_figure(fig, broadband_fig_path)
save_figure(fig, FIG_DIR / "chapter7_eegnet_broadband_interpretability.pdf")

plt.show()

print("Saved:", broadband_fig_path)

## Band-limited EEGNet attribution diagnostic

Method:
- isolate each frequency band in the EEG input
- compute EEGNet integrated gradients on the band-limited signal
- summarize attribution over time and scalp

In [ ]:
def get_safe_bands(bands: dict[str, tuple[float, float]], sfreq: float):
    nyquist_safe = sfreq / 2.0 - 1.0
    safe = {}

    for name, (low, high) in bands.items():
        high_adj = min(float(high), float(nyquist_safe))

        if high_adj <= float(low):
            print(f"[WARN] Skipping invalid band after Nyquist adjustment: {name}")
            continue

        safe[name] = (float(low), float(high_adj))

    return safe


BANDS_SAFE = get_safe_bands(BANDS, sfreq)

print("Bands requested:", BANDS)
print("Bands used     :", BANDS_SAFE)


def bandpass_epochs(
        x_np: np.ndarray,
        sfreq: float,
        low: float,
        high: float,
        order: int = 4,
) -> np.ndarray:
    nyq = sfreq / 2.0

    if high >= nyq:
        high = nyq - 1.0

    sos = butter(
        order,
        [low / nyq, high / nyq],
        btype="bandpass",
        output="sos",
    )

    return sosfiltfilt(
        sos,
        x_np,
        axis=-1,
    ).astype(np.float32)

In [ ]:
band_maps = {}
band_importance_rows = []
band_top_channel_rows = []

for band_name, (f_lo, f_hi) in BANDS_SAFE.items():
    print("=" * 80)
    print(f"Band: {band_name} ({f_lo}-{f_hi} Hz)")

    # Bandpass raw selected EEG first.
    x_band_raw = bandpass_epochs(
        X_sel_raw,
        sfreq=sfreq,
        low=f_lo,
        high=f_hi,
    )

    # Then normalize using the same train-fold stats as the EEGNet model.
    x_band_norm = normalize_epochs(
        x_band_raw,
        norm_stats,
    )

    band_attr_batches = []

    for start in range(0, len(x_band_norm), ATTR_BATCH_SIZE):
        xb = x_band_norm[start:start + ATTR_BATCH_SIZE]

        attr = integrated_gradients_batch(
            model,
            xb,
            times=times,
            steps=IG_STEPS,
        )

        band_attr_batches.append(attr)

    band_attr_trials = np.concatenate(band_attr_batches, axis=0)
    band_abs_trials = np.abs(band_attr_trials)

    band_map = band_abs_trials.mean(axis=0)
    band_maps[band_name] = band_map

    trial_strength = band_abs_trials.reshape(len(band_abs_trials), -1).mean(axis=1)

    band_importance_rows.append({
        "band": band_name,
        "f_low": f_lo,
        "f_high": f_hi,
        "importance_mean": float(trial_strength.mean()),
        "importance_std": float(trial_strength.std(ddof=1)) if len(trial_strength) > 1 else 0.0,
        "n_trials": int(len(trial_strength)),
    })

    band_p300 = band_map[:, p300_mask].mean(axis=1)

    top_band_df = (
        pd.DataFrame({
            "band": band_name,
            "channel": ch_names,
            "importance_mean": band_p300,
        })
        .sort_values("importance_mean", ascending=False)
        .head(12)
        .reset_index(drop=True)
    )

    band_top_channel_rows.append(top_band_df)

band_importance_summary = (
    pd.DataFrame(band_importance_rows)
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

band_top_channels = pd.concat(band_top_channel_rows, ignore_index=True)

save_dataframe(
    band_importance_summary,
    TABLE_DIR / "eegnet_band_importance_summary.csv",
)

save_dataframe(
    band_top_channels,
    TABLE_DIR / "eegnet_band_top_channels.csv",
)

display(round_numeric_columns(band_importance_summary, 6))

## Plot band-limited EEGNet diagnostic figure

In [ ]:
n_bands = len(band_maps)

fig, axes = plt.subplots(
    2,
    n_bands,
    figsize=(5 * n_bands, 8),
    squeeze=False,
)

for col, (band_name, band_map) in enumerate(band_maps.items()):
    band_time = band_map.mean(axis=0)
    band_p300 = band_map[:, p300_mask].mean(axis=1)

    axes[0, col].plot(times, band_time, linewidth=2)
    axes[0, col].axvspan(P300_WINDOW[0], P300_WINDOW[1], alpha=0.2)
    axes[0, col].axvline(0.0, linestyle="--", linewidth=1)
    axes[0, col].set_title(f"{band_name.capitalize()} time importance")
    axes[0, col].set_xlabel("Time (s)")
    axes[0, col].set_ylabel("Mean |attribution|")
    axes[0, col].grid(alpha=0.25)

    plot_topomap_or_text(
        axes[1, col],
        band_p300,
        info_valid,
        topomap_picks,
        f"{band_name.capitalize()} topography\n{P300_WINDOW[0]:.2f}–{P300_WINDOW[1]:.2f} s",
    )

fig.suptitle(
    f"EEGNet band-limited attribution diagnostic — {cfg.dataset.dataset}, {cfg.dataset.task}\n"
    f"Subject {REPRESENTATIVE_SUBJECT}, fold {REPRESENTATIVE_FOLD}",
    fontsize=15,
    y=1.03,
)

fig.tight_layout()

band_fig_path = FIG_DIR / "chapter7_eegnet_band_limited_diagnostic.png"

save_figure(fig, band_fig_path)
save_figure(fig, FIG_DIR / "chapter7_eegnet_band_limited_diagnostic.pdf")

plt.show()

print("Saved:", band_fig_path)

## Trial-consistency summary

Because this notebook interprets one held-out grouped fold, a trial-consistency figure is more appropriate than a subject-consistency figure.

In [ ]:
trial_time_matrix = abs_attr_trials.mean(axis=1)

trial_rows = []

for i in range(len(trial_time_matrix)):
    row = trial_time_matrix[i]

    trial_rows.append({
        "trial_index_within_selection": i,
        "subject": str(subjects_sel[i]),
        "true_label": int(y_sel[i]),
        "predicted_label": int(preds_sel[i]),
        "positive_probability": float(probs_sel[i]),
        "peak_time_s": float(times[np.argmax(row)]),
        "peak_importance": float(row.max()),
        "mean_p300_attribution": float(row[p300_mask].mean()),
    })

trial_summary = pd.DataFrame(trial_rows)

save_dataframe(
    trial_summary,
    TABLE_DIR / "eegnet_trial_attribution_summary.csv",
)

display(round_numeric_columns(trial_summary.head(), 6))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

im = axes[0].imshow(
    trial_time_matrix,
    aspect="auto",
    origin="lower",
    extent=[times[0], times[-1], 0, len(trial_time_matrix)],
)

axes[0].axvspan(P300_WINDOW[0], P300_WINDOW[1], alpha=0.2)
axes[0].axvline(0.0, linestyle="--", linewidth=1)
axes[0].set_title("Trial-by-time EEGNet attribution heatmap")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Selected trial index")

plt.colorbar(im, ax=axes[0])

axes[1].plot(
    times,
    trial_time_matrix.mean(axis=0),
    linewidth=2,
)

axes[1].axvspan(P300_WINDOW[0], P300_WINDOW[1], alpha=0.2)
axes[1].axvline(0.0, linestyle="--", linewidth=1)
axes[1].set_title("Mean attribution across selected held-out trials")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Mean |attribution|")
axes[1].grid(alpha=0.25)

fig.suptitle(
    f"EEGNet trial-consistency summary — {cfg.dataset.dataset}, {cfg.dataset.task}\n"
    f"Subject {REPRESENTATIVE_SUBJECT}, fold {REPRESENTATIVE_FOLD}",
    y=1.03,
)

fig.tight_layout()

trial_fig_path = FIG_DIR / "chapter7_eegnet_trial_consistency.png"

save_figure(fig, trial_fig_path)
save_figure(fig, FIG_DIR / "chapter7_eegnet_trial_consistency.pdf")

plt.show()

print("Saved:", trial_fig_path)

In [ ]:
run_info_json = {
    "experiment_key": EXPERIMENT_NAME,
    "dataset": cfg.dataset.dataset,
    "task": cfg.dataset.task,
    "signal": cfg.signal.slug(),
    "checkpoint_path": str(ckpt_path),
    "device": str(DEVICE),
    "heldout_subject": run_info["subject"],
    "heldout_fold": run_info["fold"],
    "seed": run_info["seed"],
    "best_epoch": run_info["best_epoch"],
    "best_val_auc": run_info["best_val_auc"],
    "loaded_from_checkpoint": run_info["loaded_from_checkpoint"],
    "positive_label": POSITIVE_LABEL,
    "use_only_correct": USE_ONLY_CORRECT,
    "ig_steps": IG_STEPS,
    "baseline_mode": BASELINE_MODE,
    "prestim_window": list(PRESTIM_WINDOW),
    "p300_window": list(P300_WINDOW),
    "bands_requested": BANDS,
    "bands_used": BANDS_SAFE,
    "n_eval_trials": int(len(X_eval_norm)),
    "n_selected_trials": int(len(X_sel_norm)),
    "eval_metrics": metrics_eval,
    "artifact_dir": str(OUT_DIR),
    "figure_dir": str(FIG_DIR),
    "table_dir": str(TABLE_DIR),
    "checkpoint_dir": str(CHECKPOINT_DIR),
}

save_json(
    run_info_json,
    RUN_INFO_JSON,
)

print("Saved:", RUN_INFO_JSON)

## Save run metadata

In [ ]:
run_info_json = {
    "experiment_key": EXPERIMENT_NAME,
    "dataset": cfg.dataset.dataset,
    "task": cfg.dataset.task,
    "signal": cfg.signal.slug(),
    "checkpoint_path": str(ckpt_path),
    "device": str(DEVICE),
    "heldout_subject": run_info["subject"],
    "heldout_fold": run_info["fold"],
    "seed": run_info["seed"],
    "best_epoch": run_info["best_epoch"],
    "best_val_auc": run_info["best_val_auc"],
    "loaded_from_checkpoint": run_info["loaded_from_checkpoint"],
    "positive_label": POSITIVE_LABEL,
    "use_only_correct": USE_ONLY_CORRECT,
    "ig_steps": IG_STEPS,
    "baseline_mode": BASELINE_MODE,
    "prestim_window": list(PRESTIM_WINDOW),
    "p300_window": list(P300_WINDOW),
    "bands_requested": BANDS,
    "bands_used": BANDS_SAFE,
    "n_eval_trials": int(len(X_eval_norm)),
    "n_selected_trials": int(len(X_sel_norm)),
    "eval_metrics": metrics_eval,
    "artifact_dir": str(OUT_DIR),
    "figure_dir": str(FIG_DIR),
    "table_dir": str(TABLE_DIR),
    "checkpoint_dir": str(CHECKPOINT_DIR),
}

save_json(
    run_info_json,
    RUN_INFO_JSON,
)

print("Saved:", RUN_INFO_JSON)

## Artifact checklist

In [ ]:
manifest = {
    "experiment_key": EXPERIMENT_NAME,
    "dataset": cfg.dataset.dataset,
    "task": cfg.dataset.task,
    "signal": cfg.signal.slug(),
    "heldout_subject": run_info["subject"],
    "heldout_fold": run_info["fold"],
    "seed": run_info["seed"],
    "artifact_dir": str(OUT_DIR),
    "figure_dir": str(FIG_DIR),
    "table_dir": str(TABLE_DIR),
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "figures": sorted([p.name for p in FIG_DIR.glob("*")]),
    "tables": sorted([p.name for p in TABLE_DIR.glob("*")]),
    "checkpoints": sorted([p.name for p in CHECKPOINT_DIR.glob("*")]),
}

save_json(
    manifest,
    OUT_DIR / "manifest.json",
)

save_experiment_metadata(
    cfg,
    extra={
        "notebook": "11_eegnet_interpretability.ipynb",
        "status": "completed",
        "experiment_key": EXPERIMENT_NAME,
        "artifact_manifest": manifest,
    },
)

print("Figures:")
for name in manifest["figures"]:
    print(" -", name)

print("\nTables:")
for name in manifest["tables"]:
    print(" -", name)

print("\nCheckpoints:")
for name in manifest["checkpoints"]:
    print(" -", name)

## Suggested thesis mapping

Recommended use:

- `chapter7_eegnet_broadband_interpretability.png`
  - main EEGNet interpretability figure

- `chapter7_eegnet_band_limited_diagnostic.png`
  - EEGNet-specific band diagnostic
  - useful for theta / alpha / gamma discussion

- `chapter7_eegnet_trial_consistency.png`
  - helpful supplementary figure
  - shows whether attribution is stable across held-out trials

Useful tables:

- `eegnet_broadband_top_channels.csv`
- `eegnet_band_importance_summary.csv`
- `eegnet_trial_attribution_summary.csv`